In [ ]:
import torch
import random
import numpy as np
from PIL import Image
from tqdm import tqdm
from pathlib import Path
from torch.utils.data import Dataset
import torchvision
from torchvision import transforms
from torchvision.models import inception_v3


class ImgDataset(Dataset):
    def __init__(self, img_paths:list, transform:torchvision.transforms=None, return_type='pil'):
        """
        Args:
            img_paths (list): 画像パスのリスト
            transform (torchvision.transforms, optional): 適用する変換
            return_type (str, optional): 返す画像の形式 ('pil', 'numpy', 'tensor')
        """
        self.img_paths = [Path(img_path) for img_path in img_paths if Path(img_path).exists()]
        self.transform = transform
        self.return_type = return_type.lower()
        if self.return_type not in ['pil', 'numpy', 'tensor']:
            raise ValueError("return_type must be one of 'pil', 'numpy', or 'tensor'")

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        
        # PIL 形式で画像を読み込む
        img_pil = Image.open(img_path).convert("RGB")
        
        # transform が指定されている場合は適用
        if self.transform:
            img = self.transform(img_pil)
        else:
            img = img_pil
        
        # 指定された形式に変換して返す
        if self.return_type == 'pil':
            if isinstance(img, torch.Tensor):
                return torchvision.transforms.ToPILImage()(img)
            elif isinstance(img, np.ndarray):
                return Image.fromarray(img)
            return img
        
        elif self.return_type == 'numpy':
            if isinstance(img, torch.Tensor):
                return img.cpu().numpy()
            elif isinstance(img, Image.Image):
                return np.array(img)
            return img
        
        elif self.return_type == 'tensor':
            if isinstance(img, np.ndarray):
                return torch.from_numpy(img)
            elif isinstance(img, Image.Image):
                return torchvision.transforms.ToTensor()(img)
            return img
    
    # 後方互換性のためのメソッド
    def get_numpy_img(self, idx):
        """numpy 形式の画像を取得"""
        temp_return_type = self.return_type
        self.return_type = 'numpy'
        img = self.__getitem__(idx)
        self.return_type = temp_return_type
        return img

def get_Inceptionv3_features(dataset):
    # Use GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Load model with pretrained weights
    model = inception_v3(weights='IMAGENET1K_V1')
    model = model.eval().to(device)
    
    # Save original transform to restore it later
    original_transform = dataset.transform
    
    # Set new transform for InceptionV3
    dataset.transform = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    features = []
    # Use tqdm for progress bar
    for i in tqdm(range(len(dataset)), desc="Extracting features"):
        img = dataset[i].unsqueeze(0).to(device)
        
        with torch.no_grad():
            # Get the output of the last layer before the final classification
            feature = model(img)
        
        features.append(feature.squeeze().cpu().numpy())
    
    # Restore original transform
    dataset.transform = original_transform
    
    return torch.tensor(features)


SyntaxError: invalid syntax (1233999770.py, line 73)

In [15]:
img_dataset_dir = Path("../../sample_data/coco_sample_datasets/sample_coco_train2017/")

sample_img_pahts_1 = random.sample(list(map(str, list(img_dataset_dir.glob("*.jpg")))), 100)
sample_img_pahts_2 = random.sample(list(map(str, list(img_dataset_dir.glob("*.jpg")))), 100)

img_dataset1 = ImgDataset(sample_img_pahts_1, transform=None, return_type='tensor')
img_dataset2 = ImgDataset(sample_img_pahts_2, transform=None, return_type='tensor')

# Use the already created datasets
print(f"Dataset 1 size: {len(img_dataset1)}")
print(f"Dataset 2 size: {len(img_dataset2)}")

inception_features1 = get_Inceptionv3_features(img_dataset1)
inception_features2 = get_Inceptionv3_features(img_dataset2)

# The features are already calculated, so we can use them directly
print(f"Feature 1 shape: {inception_features1.shape}")
print(f"Feature 2 shape: {inception_features2.shape}")

# Display sample of feature vectors
print("Sample from feature set 1:")
print(inception_features1[:2])

Dataset 1 size: 100
Dataset 2 size: 100


Extracting features: 100%|██████████| 100/100 [00:02<00:00, 45.21it/s]

Feature 1 shape: torch.Size([100, 1000])
Feature 2 shape: torch.Size([100, 1000])
Sample from feature set 1:
tensor([[ 0.3394, -0.5591,  1.2972,  ...,  0.6445,  0.1571, -0.6404],
        [ 0.4226, -0.5280, -0.5425,  ...,  0.5905,  0.9297,  0.8110]])


In [ ]:

# DreamSimネットワークを使用して特徴ベクトルを抽出
from dreamsim import dreamsim
def get_DreamSim_features(dataset: ImageDataset):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    ds_model, ds_preprocess = dreamsim(pretrained=True, cache_dir="../__model__/00_misc/DreamSim/", device=device)
    
    features = []
    for idx in tqdm(range(len(dataset))):
        # torch.tensor to PIL.Image
        img = Image.fromarray(dataset.get_numpy_img(idx))
        img = ds_preprocess(img).to(device)
        features.append(ds_model.embed(img).detach().cpu().squeeze().numpy())
        
    return torch.tensor(features)
    